# Chapter 13: Distributed Transactions

*From: Database Internals*

## TL;DR

- A **distributed transaction** must commit atomically across multiple partitions. Single-partition concurrency control (2PL, MVCC) doesn't solve this — we need an **atomic commitment** protocol that unanimously agrees to commit or abort.
- **2PC** (propose → commit/abort) is the workhorse: simple, low message count, but **blocking** if the coordinator dies mid-round. **3PC** adds a `prepare` step and per-cohort timeouts so cohorts can decide without the coordinator, but it is **not safe under network partitions** — split brain is real.
- Three architectural families dominate modern systems: **Calvin** preorders all transactions via a Paxos-replicated sequencer so 2PC is unnecessary; **Spanner** runs classical 2PC on top of per-shard Paxos groups and uses **TrueTime commit-wait** to guarantee external consistency; **Percolator** builds client-driven 2PC with snapshot isolation on top of Bigtable using lock/data/write columns.
- **Consistent hashing** is the partitioning layer underneath much of this: on cluster resize, only `K/n` keys move to immediate neighbors instead of most keys reshuffling as in `hash mod N`.
- **Coordination avoidance** (I-Confluence) and **RAMP** attack the problem from the other direction: if invariants are mergeable, skip coordination entirely; RAMP gives you read-atomic isolation across partitions without mutual exclusion.

## Problem Framing

Single-node transactions get atomicity "for free" from the WAL: one fsync and the commit is durable and visible. Distributed transactions can't. When a transaction touches partitions on nodes `A`, `B`, `C`, three things can break simultaneously:

1. **Multipartition atomicity.** Either *all* of `A`, `B`, `C` apply the writes or *none* do. No partial visibility. No node that committed locally but lost the network may reveal its state.
2. **Network partitions.** Any node can become unreachable at any time. Messages can be delayed arbitrarily, reordered, or dropped.
3. **Coordinator failures.** Whichever process is orchestrating the commit is itself a potentially-failing node. Its death mid-round can leave cohorts holding locks and waiting forever.

Atomic commitment algorithms are about reaching unanimous agreement among cohorts on commit-vs-abort, surviving (1) and (3), while making honest trade-offs around (2). They explicitly assume **non-Byzantine** processes — cohorts don't lie about their vote.

## Back-of-Envelope Estimation

Two concrete numbers: message complexity of each commit family, and the latency tax Spanner pays for external consistency via TrueTime commit-wait.

In [ ]:
# ----- Part 1. Message complexity for N cohorts -----
def msg_counts(n_cohorts: int) -> dict[str, int]:
    # 2PC: coordinator sends Propose to N, N reply, coordinator sends Commit to N,
    # N ack. Two round trips, 4N messages on the happy path.
    two_pc = 4 * n_cohorts

    # 3PC: Propose/vote + Prepare/ack + Commit/ack. Three round trips, 6N messages.
    three_pc = 6 * n_cohorts

    # Calvin: the cohorts don't talk a commit protocol at all on the hot path.
    # The sequencer broadcasts the batch once via Paxos (2-phase Paxos across R
    # replicas of the log). Per transaction, within a batch of B txns, amortized
    # msgs_per_txn ~= (paxos_rounds * R) / B. For B=100 txns, R=3 replicas:
    R, B = 3, 100
    paxos_round_msgs = 2 * R   # prepare + accept quorum, amortized
    calvin = paxos_round_msgs / B + n_cohorts  # + local scheduler forwards
    return {"2PC": two_pc, "3PC": three_pc, "Calvin (amortized/txn)": round(calvin, 2)}

for n in (2, 5, 10, 50):
    print(f"N={n:>2} cohorts  -> {msg_counts(n)}")

# ----- Part 2. Spanner commit-wait overhead from TrueTime uncertainty -----
# TrueTime exposes [earliest, latest] with uncertainty epsilon ~ few ms.
# Commit-wait: after picking commit ts = TT.now().latest, the leader sleeps
# until TT.now().earliest > commit_ts. Expected wait ~= epsilon.
epsilon_ms_typical = 7.0      # Google published ~1-7ms after GPS/atomic fix
epsilon_ms_worst = 14.0       # post-reconfiguration or bad clock
txn_latency_wan_ms = 50.0     # 2PC over Paxos round trips in a region

for eps in (epsilon_ms_typical, epsilon_ms_worst):
    total = txn_latency_wan_ms + eps
    tax_pct = 100 * eps / total
    print(f"epsilon={eps:>4.1f} ms  commit_total={total:>5.1f} ms  "
          f"commit-wait tax = {tax_pct:.1f}%")

# Throughput intuition: if one leader can serialize commits, upper bound QPS
# per leader ~= 1000 / (txn_latency + epsilon).
for eps in (epsilon_ms_typical, epsilon_ms_worst):
    qps = 1000 / (txn_latency_wan_ms + eps)
    print(f"epsilon={eps:>4.1f} ms  per-leader serial commit QPS ~= {qps:.0f}")

The message-count table is the honest picture: **2PC is cheapest**, 3PC pays a 50% message premium for its (fragile) non-blocking property, and **Calvin amortizes consensus across a batch** so per-transaction messaging is dominated by the local forwards, not the log. For Spanner, a ~7 ms epsilon adds ~12% to a 50 ms WAN transaction — real but tolerable; a bad-clock 14 ms epsilon is where external consistency starts to visibly cost you.

## Atomic Commitment: 2PC and 3PC

Both protocols assume a **coordinator** plus N cohorts, persistent logs on every participant, and (for 3PC) a synchronous-enough network.

### Two-Phase Commit

```mermaid
sequenceDiagram
  participant C as Coordinator
  participant P1 as Cohort 1
  participant P2 as Cohort 2
  participant P3 as Cohort 3
  C->>P1: Propose(tx)
  C->>P2: Propose(tx)
  C->>P3: Propose(tx)
  P1-->>C: Yes (precommit, log)
  P2-->>C: Yes (precommit, log)
  P3-->>C: Yes (precommit, log)
  Note over C: All yes -> decide Commit. Log decision first.
  C->>P1: Commit
  C->>P2: Commit
  C->>P3: Commit
  P1-->>C: Ack
  P2-->>C: Ack
  P3-->>C: Ack
```

**The blocking problem.** If the coordinator logs "all voted yes, I decide Commit" but crashes *before* broadcasting, every cohort sits holding locks and cannot progress unilaterally. Rule: a cohort that voted yes promised not to unabort — so it cannot timeout and abort (another cohort might already have received a Commit from the dead coordinator). It *must wait* for the coordinator to recover or for a recovery coordinator to read the decision log and finish the round. Meanwhile, those locks block unrelated transactions. This is the single biggest practical pain point of 2PC.

A single cohort failure during propose is less scary: the coordinator times out waiting for a vote and aborts. But a failed cohort that *did* vote yes before crashing must, on recovery, fetch the decision from the coordinator (or peer logs) before it can resume serving.

### Three-Phase Commit

```mermaid
sequenceDiagram
  participant C as Coordinator
  participant P as Cohorts
  C->>P: Propose(tx)
  P-->>C: Vote yes/no (persist)
  Note over C: If all yes -> move to Prepare
  C->>P: Prepare (results of vote phase)
  P-->>C: Prepare-Ack (persist "prepared")
  Note over C,P: Now every prepared cohort knows all others voted yes.
  C->>P: Commit
  P-->>C: Committed
```

3PC adds the `Prepare` phase so that, before the final `Commit`, **all cohorts know every other cohort has voted yes**. That lets a cohort that has reached "prepared" state safely commit unilaterally after a timeout — nobody else could have decided differently. Cohorts that never made it to `prepared` abort on timeout.

**Split-brain risk.** 3PC only works under synchronous network assumptions. In a partition, some cohorts reach `prepared` (behind the partition with the coordinator) and commit on timeout; others never hear the `Prepare`, timeout, and abort. Now some partitions have committed the transaction and others have aborted it — a protocol-legal, permanently-inconsistent split brain. This is why 3PC is rarely used in practice; modern systems either accept 2PC's blocking behavior plus add a backup coordinator, or switch to consensus-based protocols where failure detection and leader election are built-in.

## Deterministic Transactions: Calvin

Calvin sidesteps 2PC by removing the question 2PC is designed to answer. Instead of letting cohorts execute transactions concurrently and then voting on whether to commit, Calvin **decides the total order of transactions first**, replicates that order via Paxos, and then has every replica execute the same sequence deterministically. Every replica will reach the same post-state, so there's nothing to vote on.

```mermaid
graph TD
  subgraph "Replica 1 (same layout on every replica)"
    Seq1[Sequencer: collect 10ms epoch batch]
    Sched1[Scheduler: parse read/write sets, pick dependency order]
    Wk1[Worker threads: execute active partitions]
    St1[(Storage: local tablets)]
    Seq1 --> Sched1 --> Wk1 --> St1
  end
  Client --> Seq1
  Seq1 <-- "Paxos: agree on batch per epoch" --> Seq2[Sequencer R2]
  Seq1 <-- "Paxos" --> Seq3[Sequencer R3]
  Wk1 <-- "forward read-set rows between active nodes" --> WkX[Workers on other partitions]
```

**How it sidesteps 2PC.** Cohorts don't ask each other "can you commit?" because the transaction's input (read set, write set) is already known and the order is already agreed. Workers on "active" nodes (those holding write-set rows) receive the forwarded read-set rows, execute locally, and persist. If one worker crashes, another replica of the same partition already has the deterministic inputs and produces the identical output — recovery is replay, not rollback.

**The catch.** Calvin requires the **read/write sets to be known upfront**. Transactions that must read a value to decide what to read next (common in OLTP, e.g., "read order.shipping_id, then look up that shipment") need a `reconnaissance` pre-read and a validation step, adding latency. Sequencer batching (10 ms epochs in the paper) also means per-transaction latency is lower-bounded by the batch window.

## Spanner: 2PC over Paxos with TrueTime

Spanner takes the opposite bet: keep classical 2PC, but fix its fragility by making each cohort a **Paxos group** (so a single-node failure doesn't kill the round), and fix clock-based ordering by paying for **bounded-uncertainty physical clocks (TrueTime)**.

```mermaid
graph TD
  Client --> TxMgr[Client-side transaction manager]
  TxMgr -- "read at latest / at ts" --> L1[Leader of Paxos group G1]
  TxMgr -- "writes, 2PC prepare" --> L1
  TxMgr -- "writes, 2PC prepare" --> L2[Leader of Paxos group G2]
  TxMgr -- "writes, 2PC prepare" --> L3[Leader of Paxos group G3]
  subgraph "Paxos Group G1 (one shard)"
    L1 --- R1a[Replica]
    L1 --- R1b[Replica]
    L1 --> T1[(Tablet: versioned rows)]
    R1a --> T1a[(Tablet)]
    R1b --> T1b[(Tablet)]
  end
  L1 -. "commit ts >= all prepare ts" .- TxMgr
  TxMgr -. "commit-wait until TT.now().earliest > commit_ts" .- TxMgr
```

**Commit sequence for a multi-shard write:**
1. Client acquires write locks on each involved Paxos group's leader (2PL).
2. Each leader logs a **prepare** record through its Paxos group (so the "prepared" state survives leader failure) and returns a **prepare timestamp**.
3. The designated coordinator leader picks **commit_ts** strictly greater than all prepare timestamps *and* greater than `TT.now().latest`.
4. **Commit-wait:** the coordinator sleeps until `TT.now().earliest > commit_ts` before logging the commit record and releasing locks.

**External consistency.** Commit-wait guarantees that if transaction `T1` commits before `T2` starts (real-time order), then `T1.commit_ts < T2.commit_ts`. Any future reader picking a timestamp sees `T1` before `T2` — linearizability for multi-object transactions. The cost is the wait itself, averaging one epsilon (~1–7 ms on modern Spanner). Within a single shard, no 2PC is needed; a single Paxos-group commit suffices.

## Partitioning and Consistent Hashing

Both Calvin and Spanner presume data is already partitioned across nodes. The routing key → node mapping function determines how badly a cluster resize hurts.

### hash mod N vs. consistent hashing

With `hash(key) mod N`, changing N shuffles almost every key. With consistent hashing, keys and nodes both map onto a ring; each node owns the arc from the previous node clockwise to itself. Adding or removing a node only moves the keys on its immediate arcs.

```mermaid
graph TD
  subgraph "Hash ring (0 ... 2^64 - 1, wraps)"
    N1((N1 @ 0x10))
    N2((N2 @ 0x40))
    N3((N3 @ 0x80))
    N4((N4 @ 0xC0))
  end
  K1[key A hash=0x30] --> N2
  K2[key B hash=0x70] --> N3
  K3[key C hash=0xA0] --> N4
  K4[key D hash=0xF0] -->|wraps past 0x100| N1
  NewN((N5 @ 0x60 - join)) -. "steals range 0x41..0x60 from N3" .-> N3
```

| | `hash(key) mod N` | Consistent hashing |
|---|---|---|
| Data moved on resize | ~`(N-1)/N` of keys (most of the dataset) | ~`K/n` keys, only on 1–2 neighbors |
| Hotspot avoidance | Uniform (hash spreads naturally) | Needs **virtual nodes** so one physical node owns multiple arcs, or load can be lumpy |
| Adding a node | Re-hash everything, full reshuffle | Steal adjacent arc from one or two neighbors |
| Used by | Simple memcached setups | Cassandra, DynamoDB, Riak |

## Percolator: Snapshot Isolation via Client-Driven 2PC

Percolator is a transaction layer on top of Bigtable. It stores three columns for each logical cell: `data` (the value at a timestamp), `lock` (presence = row is locked by some in-flight transaction), and `write` (pointer from commit timestamp → data timestamp, i.e., the committed version history). A **timestamp oracle** hands out cluster-wide monotonic timestamps.

**Transaction protocol (client drives 2PC):**
1. At `start_ts`, read a committed snapshot (by ignoring any `write` entries with ts > `start_ts`).
2. Buffer writes in the client.
3. **Prewrite**: for each buffered write, atomically (via Bigtable conditional mutate) check no conflicting `write` exists after `start_ts` and no `lock` is already held, then place the new `data` and a `lock`. Designate one row's lock as the **primary**; all secondary locks reference the primary.
4. At `commit_ts`, commit the primary: atomically replace its `lock` with a `write` entry pointing at `start_ts`. Once this succeeds, the transaction is committed.
5. Replace each secondary lock with a corresponding `write` entry. Secondary cleanup can be lazy; readers who see a secondary lock chase the primary to decide if the transaction committed.

```mermaid
graph TD
  subgraph "(a) Initial — TS1 for both"
    A1["Acct1: data@TS1=100 write@TS1 lock=-"]
    A2["Acct2: data@TS1=200 write@TS1 lock=-"]
  end
  subgraph "(b) Prewrite at ts=TS2"
    B1["Acct1: data@TS2=250 write@TS1 lock=PRIMARY"]
    B2["Acct2: data@TS2=50  write@TS1 lock=->primary(Acct1)"]
  end
  subgraph "(c) Commit at ts=TS3 (primary first)"
    C1["Acct1: data@TS1,TS2 write@TS1,TS3->TS2 lock=-"]
    C2["Acct2: data@TS1,TS2 write@TS1,TS3->TS2 lock=-"]
  end
  A1 --> B1 --> C1
  A2 --> B2 --> C2
```

**Failure handling.** If the client crashes between step 4 and step 5, secondary rows still have locks. A later transaction that bumps into a stale secondary lock looks up the **primary**: if the primary has already been committed (has a `write` entry at `commit_ts`), the stalker helps finish the commit by cleaning up the secondaries; if the primary still shows a lock or was rolled back, the stalker aborts the secondary. This "help or abort" rule is what lets Percolator use a dumb KV layer underneath — recovery is decentralized and piggybacked on readers.

**Isolation level.** Snapshot isolation: reads see a consistent snapshot at `start_ts`, first-committer-wins prevents lost updates, but **write skew** is still allowed (disjoint write sets that individually respect an invariant can jointly violate it). Good enough for many OLTP workloads; TiDB is built on this model.

## Coordination Avoidance and RAMP

The insight behind coordination avoidance: **some operations don't need to coordinate at all**. An operation is **I-Confluent** with respect to an invariant `I` if, given any two `I`-valid states produced by concurrent execution, their merge is still `I`-valid. I-Confluent operations can run on local snapshots, commit independently, and merge later — no distributed agreement required.

Example: incrementing counters with invariant "counter ≥ 0" *is not* I-confluent under concurrent decrement; invariant "append-only log" *is* I-confluent under concurrent appends.

**RAMP (Read-Atomic Multi-Partition)** applies this to multi-partition reads. It provides **read-atomic isolation**: a reader either sees *all* writes of a transaction across all involved partitions, or *none* of them — no **fractured reads** — but without distributed locks. The trick: writers install values in a hidden, "prepared" form first, then atomically flip visibility; readers use metadata attached to every row to detect in-flight writes and fetch the latest consistent version.

```mermaid
sequenceDiagram
  participant W as Writer
  participant P1 as Partition 1
  participant P2 as Partition 2
  participant P3 as Partition 3
  W->>P1: Prepare(v1, tx_id, sibling_keys={k2,k3})
  W->>P2: Prepare(v2, tx_id, sibling_keys={k1,k3})
  W->>P3: Prepare(v3, tx_id, sibling_keys={k1,k2})
  Note over P1,P3: writes exist but not yet visible
  W->>P1: Commit(tx_id)
  W->>P2: Commit(tx_id)
  W->>P3: Commit(tx_id)
  Note over P1,P3: Readers who see (k1@tx) but not (k2@tx) chase metadata and fetch the latest from P2
```

**Guarantees.**
- **Synchronization independence** — readers and writers never block each other.
- **Partition independence** — clients skip partitions they don't touch.
- **Read-atomic** — atomic cross-partition visibility without serializability.

**Trade-off.** Weaker than serializability: write skew, lost updates in the general case, and no ordering between unrelated transactions. Good for workloads where the only multi-partition guarantee you truly need is "never see half of a transaction."

## Trade-offs and Alternatives

| Approach | Pros | Cons | When to Use |
|---|---|---|---|
| **2PC** | Simplest atomic commitment; low message count (4N); universally understood | Blocking on coordinator failure; cohort locks held through the round; hurts availability | Short transactions on a reliable network where a backup coordinator plus careful recovery is enough (MySQL XA, MongoDB distributed txns) |
| **3PC** | Non-blocking on coordinator failure if the network is synchronous | Split brain under network partitions; 50% more messages (6N) than 2PC; rarely used in practice | Mostly of theoretical interest; use a consensus protocol instead |
| **Calvin** | No on-path 2PC; replica failures are recoverable via replay; high throughput on known read/write sets | Requires upfront read/write sets (reconnaissance pre-reads otherwise); batch window adds latency floor | High-throughput OLTP where transactions are short, read/write sets are statically derivable (FaunaDB) |
| **Spanner (2PC over Paxos + TrueTime)** | External consistency (linearizable multi-object); per-shard failures tolerated via Paxos groups; rich query language | TrueTime commit-wait adds ~epsilon latency; GPS/atomic clock infrastructure or bounded-uncertainty NTP; multi-shard txns cost 2PC | Global, externally consistent SQL (Spanner, CockroachDB, YugabyteDB) |
| **Percolator** | Snapshot isolation on top of a dumb KV store; decentralized recovery via primary-lock chasing; good reader throughput | Write skew allowed (SI, not serializable); two RPCs to timestamp oracle per txn; lock/data/write overhead per row | Analytics/OLTP hybrid on an existing wide-column/KV layer (Google's web indexing, TiDB) |
| **RAMP / coordination-avoidance** | No blocking between readers and writers; partition-independent reads; high throughput at scale | Only read-atomic (not serializable); write skew, lost updates; app must validate invariants are I-confluent | Multi-partition reads where fractured reads are the only anomaly you care about (shopping cart across shards, materialized indexes) |

## Key Takeaways

1. **Atomic commitment and consensus are different problems.** 2PC/3PC try to reach unanimous agreement per-transaction; consensus (Paxos/Raft) replicates a log of decisions. Modern systems layer one on the other: 2PC on Paxos groups (Spanner) or a Paxos-replicated sequencer that obsoletes 2PC (Calvin).
2. **2PC's blocking isn't a bug, it's the guarantee.** A cohort that voted yes promised it cannot unilaterally abort — that's exactly why 2PC is safe. The price is unavailability during coordinator failure. Backup coordinators plus Paxos replication of the decision log mitigate it; they don't eliminate it.
3. **3PC is a cautionary tale.** It looks like a clean fix to 2PC blocking, but it assumes a synchronous network. Partitions break it. Don't deploy it; learn from it.
4. **Preordering (Calvin) or ordered commits on Paxos (Spanner) dominate modern designs.** Classical 2PC alone is no longer the default for new systems — it's always combined with or replaced by consensus.
5. **TrueTime is the clearest example of buying a primitive with infrastructure.** Commit-wait is cheap if your hardware can keep epsilon small; every distributed SQL that wants external consistency without GPS clocks has to reinvent a software analog (CockroachDB's HLCs with uncertainty intervals, for example).
6. **Partitioning shapes the whole problem.** Consistent hashing is the default underneath Cassandra-class systems; range partitioning underpins Spanner and Percolator. The choice affects failure blast radius, rebalancing cost, and range-scan friendliness.
7. **Coordination is expensive — avoid it where the invariants let you.** I-Confluence gives a formal test for when. RAMP is the classic "strong-enough-but-cheap" design for multi-partition visibility without serializability.

## See Also

- [[01-two-phase-commit]]
- [[02-three-phase-commit]]
- [[03-calvin-deterministic-transactions]]
- [[04-spanner-truetime]]
- [[05-consistent-hashing]]
- [[06-percolator-snapshot-isolation]]
- [[07-coordination-avoidance-ramp]]